# Задание 2.

## Этап 3. Реализация наследования
Реализуйте два класса CheckingAccount (расчётный счёт) и SavingsAccount (сберегательный счёт), которые отражают абстракцию базового поведения банковских аккаунтов:

● наследуются от базового класса Account;

● хранят атрибут класса account_type.

● Класс SavingsAccount (сберегательный счёт) дополнительно должен реализовывать метод расчёта процентов на остаток apply_interest(self, rate) (например, 7% на остаток).

● Класс SavingsAccount (сберегательный счёт) позволяет снимать деньги только до определенного порога баланса: нельзя снять больше 50% от баланса. Переопределите метод снятия со счёта.

Реализуйте валидацию на отрицательные суммы и корректность имени владельца:

● имя владельца счёта должно быть в формате «Имя Фамилия» с заглавных букв, кириллицей или латиницей, иначе — должно вызываться исключение;

● попытка положить отрицательную сумму должна вызывать исключение.

Реализуйте метод для анализа истории транзакций по размеру и дате:

● метод должен выводить последние n крупных операций.

In [2]:
from datetime import datetime
from typing import Any
import re

class Account:
    accounts_number_order: int = 1

    def __init__(self, account_holder: str, balance: float = 0):
        self.account_holder = None

        self.set_account_holder(account_holder)

        self.account_number: str = "ACC-" + str(Account.accounts_number_order).zfill(4)
        self.__operations_history: dict = {}
        self.__account_counter = 1000
        self._balance: float = 0

        self.deposit(balance)

        Account.accounts_number_order += 1

    def get_balance(self) -> float:
        return self._balance

    def get_operations_history(self) -> str:
        """
        Получение истории операций

        :return: str
        """
        result = ''

        for date, operations in self.__operations_history.items():
            if len(operations) == 0:
                continue

            result += f"{date}:"

            for operation in operations:
                result += "\n   " + " | ".join(f"{k}: {v}" for k, v in operation.items())

        return result

    def deposit(self, amount: float) -> bool:
        """
        Внесение суммы на счет

        :param amount:
        :raises ValueError:
        :return: True
        """
        status = amount >= 0

        if status:
            self._balance += amount

        self.log_new_operation('deposit', amount, status)

        if not status:
            raise ValueError('Сумма должна быть не отрицательной!')

        return status

    def withdraw(self, amount: float) -> bool:
        """
        Снятие денег со счета

        :param amount:
        :raises ValueError:
        :return:
        """
        error: str | False = False

        if amount < 0:
            error = 'Сумма должна быть положительной!'

        elif amount > self.get_balance():
            error = 'Сумма должна быть меньше текущего баланса!'

        else:
            self._balance -= amount

        self.log_new_operation('withdraw', amount, not bool(error))

        if error:
            raise ValueError(error)

        return True

    def log_new_operation(self, operation_type: str, data: Any, status: bool) -> None:
        """
        Сохранение новой операции

        :param operation_type:
        :param data:
        :param status:
        :return:
        """
        current_date = datetime.now()
        date = current_date.strftime("%d-%m-%Y")
        time = current_date.strftime("%H:%M:%S")

        self.__operations_history.setdefault(date, []).append({
            'time': time,
            'operation_type': operation_type,
            'status': 'success' if status else 'fail',
            'balance': self.get_balance(),
            'data': data,
        })

    def set_account_holder(self, name: str) -> None:
        """
        Валидация и присваивание значения в поле account_holder

        :param name:
        :raises ValueError:
        :return:
        """
        parts = name.split()

        if len(parts) != 2 or " " * 2 in parts:
            raise ValueError("Имя владельца должно быть в формате: 'Имя Фамилия'")

        test = name.replace('-', '')

        if not test.istitle():
            raise ValueError("Имя и фамилия должны быть с большой буквы, и только они")

        if not test.replace(' ', '').isalpha():
            raise ValueError("Цифры и спец. символы (кроме дефиса) не допустимы")

        if re.search(r'[а-яёА-ЯЁ]', test) and re.search(r'[a-zA-Z]', test):
            raise ValueError("Смешивание кириллицы и латиницы не допустимо")

        self.account_holder = name

    def get_last_large_operations(self, count: int, by_date: str | None = None) -> str:
        """
        Получение последних крупных операций

        :param count: Выводимое кол-во операций
        :param by_date: Опционально - по дате
        :return:
        """
        result = ''

        i = count

        for date, operations in self.__operations_history.items():
            if i <= 0:
                break

            if by_date and by_date not in date:
                continue

            large_date_operations = ''

            for operation in operations:
                if i <= 0:
                    break

                if not isinstance(operation['data'], int | float) or operation['data'] < 1000:
                    continue

                large_date_operations += "\n   " + " | ".join(f"{k}: {v}" for k, v in operation.items())
                i -= 1

            if len(large_date_operations) > 0:
                result += f"{date}:{large_date_operations}"

        return result


class CheckingAccount(Account):
    account_type = "Расчетный счет"


class SavingsAccount(Account):
    account_type = "Сберегательный счет"

    def apply_interest(self, rate: int) -> float:
        """
        Расчет и применение процентов на остаток

        :param rate:
        :return:
        """
        interest = self.get_balance() * (rate / 100)

        self._balance += interest

        self.log_new_operation('apply_interest', interest, True)

        return interest

    def withdraw(self, amount: float) -> bool:
        """
        Переопределение метода снятия со счета. Не позволяет снять больше 50% от текущей суммы

        :param amount:
        :raises ValueError:
        :return:
        """
        error: str | False = False

        if amount < 0:
            error = 'Сумма должна быть положительной!'

        elif amount > self.get_balance():
            error = 'Сумма должна быть меньше текущего баланса!'

        elif amount > self.get_balance() // 2:
            error = 'Сумма не должна быть больше 50% текущего баланса!'

        else:
            self._balance -= amount

        self.log_new_operation('withdraw', amount, not bool(error))

        if error:
            raise ValueError(error)

        return True


c = CheckingAccount('Тест Тестов', 1000)

c.withdraw(100)
c.withdraw(700)

c.deposit(5000)
c.deposit(3000)

c.withdraw(2400)

print(c.get_operations_history())
print(c.get_last_large_operations(4, '02-10-2026'))



02-10-2026:
   time: 22:29:09 | operation_type: deposit | status: success | balance: 1000 | data: 1000
   time: 22:29:09 | operation_type: withdraw | status: success | balance: 900 | data: 100
   time: 22:29:09 | operation_type: withdraw | status: success | balance: 200 | data: 700
   time: 22:29:09 | operation_type: deposit | status: success | balance: 5200 | data: 5000
   time: 22:29:09 | operation_type: deposit | status: success | balance: 8200 | data: 3000
   time: 22:29:09 | operation_type: withdraw | status: success | balance: 5800 | data: 2400
02-10-2026:
   time: 22:29:09 | operation_type: deposit | status: success | balance: 1000 | data: 1000
   time: 22:29:09 | operation_type: deposit | status: success | balance: 5200 | data: 5000
   time: 22:29:09 | operation_type: deposit | status: success | balance: 8200 | data: 3000
   time: 22:29:09 | operation_type: withdraw | status: success | balance: 5800 | data: 2400
